# Role-Based Access Control — Deploy & Test

This notebook deploys an AgentCore agent + MCP server stack and verifies that
**role-based access control (RBAC)** works end-to-end.

### What gets deployed
- **Cognito User Pool** with two pre-created users (`user-a` as FinanceUser, `user-b` as HRUser)
- **Agent Runtime** (Strands agent with calculator + MCP tools)
- **MCP Server Runtime** (FastMCP with role-gated tools)
- **Pre-token Lambda** that copies `custom:roles` into Cognito access tokens
- **Secrets Manager** secret for M2M client credentials

### What gets tested
| User | Role | `get_stock_price` | `get_employee_count` | Public tools |
|------|------|:-:|:-:|:-:|
| user-a | FinanceUser | ✅ Allowed | ❌ Denied | ✅ |
| user-b | HRUser | ❌ Denied | ✅ Allowed | ✅ |

### Prerequisites
- AWS account with Bedrock AgentCore access
- AWS credentials configured (`aws configure`)
- Docker installed and running
- Node.js 18+ (required by CDK)
- CDK bootstrapped (`npx cdk bootstrap`)

## Step 0: Install Dependencies

If Node.js is not installed, uncomment the line for your OS:

In [ ]:
# --- Uncomment ONE line matching your OS to install Node.js ---

# macOS (with Homebrew)
#!brew install node

# macOS (without Homebrew) — installs to ~/.local, no sudo needed
#!mkdir -p ~/.local && curl -fsSL https://nodejs.org/dist/v18.20.8/node-v18.20.8-darwin-arm64.tar.xz | tar -xJ --strip-components=1 -C ~/.local && export PATH="$HOME/.local/bin:$PATH"

# Ubuntu / Debian
#!curl -fsSL https://deb.nodesource.com/setup_18.x | sudo -E bash - && sudo apt-get install -y nodejs

# Amazon Linux / RHEL
#!curl -fsSL https://rpm.nodesource.com/setup_18.x | sudo bash - && sudo yum install -y nodejs

# Windows (run in PowerShell)
#!winget install OpenJS.NodeJS.LTS

In [ ]:
# Install Python dependencies (CDK library, boto3, requests)
!cd .. && python3 -m venv .venv && source .venv/bin/activate && pip install -q .

## Step 1: Deploy the CDK Stack

This creates all AWS resources and writes connection details to `outputs.json`.

In [17]:
# Deploy the stack — PATH includes /usr/local/bin so CDK can find Docker
!cd .. && PATH=$PWD/.venv/bin:$HOME/.local/bin:/usr/local/bin:/opt/homebrew/bin:/usr/bin:$PATH npx --yes cdk deploy --outputs-file outputs.json --require-approval never

⠙⠹⠸
✨  Synthesis time: 5.49s

AgentCoreCICDStack-dev: deploying... [1/1]
AgentCoreCICDStack-dev: creating CloudFormation changeset...
[··························································] (0/23)

3:28:05 PM | REVIEW_IN_PROGRESS   | AWS::CloudFormation::Stack            | Age
ntCoreCICDStack-dev
[··························································] (0/23)

3:28:10 PM | CREATE_IN_PROGRESS   | AWS::CloudFormation::Stack            | Age
ntCoreCICDStack-dev
[··························································] (0/23)

3:28:10 PM | CREATE_IN_PROGRESS   | AWS::CloudFormation::Stack            | Age
ntCoreCICDStack-dev
3:28:13 PM | CREATE_IN_PROGRESS   | AWS::IAM::Role                        | Pre
TokenFn/ServiceRole
[··························································] (0/23)

3:28:10 PM | CREATE_IN_PROGRESS   | AWS::CloudFormation::Stack            | Age
ntCoreCICDStack-dev
3:28:13 PM | CREATE_IN_PROGRESS   | AWS::IAM::Role                        | Pre
TokenFn/Se

## Step 2: Set User Passwords

CDK creates `user-a` and `user-b` in Cognito but leaves them in `FORCE_CHANGE_PASSWORD` status.
We need to set permanent passwords before they can authenticate.

In [ ]:
import json, boto3, getpass

# Load stack outputs
with open('../outputs.json') as f:
    outputs = json.load(f)['AgentCoreCICDStack-dev']

POOL_ID = outputs['SharedUserPoolId']
REGION = 'ap-southeast-2'

# Prompt for password — must meet Cognito policy (8+ chars, upper, lower, number, symbol)
PASSWORD = getpass.getpass('Enter password for test users: ')

# Set the same password for both test users
cog = boto3.client('cognito-idp', region_name=REGION)
for user in ['user-a', 'user-b']:
    cog.admin_set_user_password(UserPoolId=POOL_ID, Username=user, Password=PASSWORD, Permanent=True)
    print(f'Password set for {user}')

## Step 3: Wait for Containers

AgentCore needs time to start the agent and MCP server containers after deployment.

In [ ]:
import os, time

# Set environment variables used by the test code below
os.environ['AGENT_RUNTIME_ARN'] = outputs['AgentRuntimeArn']
os.environ['USER_POOL_ID'] = outputs['SharedUserPoolId']
os.environ['USER_CLIENT_ID'] = outputs['UserClientId']
os.environ['AWS_REGION'] = REGION
os.environ['USER_A_PASSWORD'] = PASSWORD
os.environ['USER_B_PASSWORD'] = PASSWORD

print('Waiting 30s for containers to start...')
time.sleep(30)
print('Ready.')

## Step 4: Run Role-Based Access Tests

Each test authenticates as a specific user, invokes the agent, and checks whether
the response contains (or doesn't contain) expected tool output.

- **ALLOWED** tests check that the tool result appears in the response
- **DENIED** tests check that the tool result does NOT appear (the tool is hidden from the user)

In [ ]:
import json, os, time, uuid, urllib.parse
import boto3, requests

# --- Configuration from environment ---
REGION = os.environ.get('AWS_REGION', 'ap-southeast-2')
AGENT_ARN = os.environ['AGENT_RUNTIME_ARN']
USER_POOL_ID = os.environ['USER_POOL_ID']
CLIENT_ID = os.environ['USER_CLIENT_ID']


def get_access_token(username: str, password: str) -> str:
    """Authenticate a Cognito user and return their access token."""
    cog = boto3.client('cognito-idp', region_name=REGION)
    resp = cog.admin_initiate_auth(
        UserPoolId=USER_POOL_ID, ClientId=CLIENT_ID,
        AuthFlow='ADMIN_NO_SRP_AUTH',
        AuthParameters={'USERNAME': username, 'PASSWORD': password},
    )
    return resp['AuthenticationResult']['AccessToken']


def invoke(prompt: str, token: str, timeout: int = 120) -> dict:
    """Invoke the agent via HTTPS with a Bearer token.
    Each call uses a unique session ID to get a fresh container."""
    escaped = urllib.parse.quote(AGENT_ARN, safe='')
    url = f'https://bedrock-agentcore.{REGION}.amazonaws.com/runtimes/{escaped}/invocations?qualifier=DEFAULT'
    r = requests.post(
        url,
        headers={
            'Authorization': f'Bearer {token}',
            'Content-Type': 'application/json',
            'X-Amzn-Bedrock-AgentCore-Runtime-Session-Id': str(uuid.uuid4()),
        },
        data=json.dumps({'prompt': prompt}),
        timeout=timeout,
    )
    return {'status': r.status_code, 'body': r.text}


def check(result: dict, should_contain=None, should_not_contain=None) -> bool:
    """Check if the response contains/excludes expected strings."""
    body = result['body'].lower()
    ok = result['status'] == 200
    for s in (should_contain or []):
        if s.lower() not in body:
            ok = False
    for s in (should_not_contain or []):
        if s.lower() in body:
            ok = False
    return ok


# --- Test matrix ---
# (user, prompt, must_contain, must_not_contain, description)
TEST_CASES = [
    # Public tools — accessible to both users
    ('user-a', 'What is the current time in UTC?', ['202'], [], 'user-a: get_current_datetime (public)'),
    ('user-b', 'How much is 15 * 7?', ['105'], [], 'user-b: calculator (public)'),

    # get_stock_price — only FinanceUser
    ('user-a', 'What is the stock price of AAPL?', ['175.50'], [], 'user-a (FinanceUser): get_stock_price ALLOWED'),
    ('user-b', 'What is the stock price of AAPL?', [], ['175.50'], 'user-b (HRUser): get_stock_price DENIED'),

    # get_employee_count — only HRUser
    ('user-b', 'How many employees are in engineering?', ['150'], [], 'user-b (HRUser): get_employee_count ALLOWED'),
    ('user-a', 'How many employees are in engineering?', [], ['150'], 'user-a (FinanceUser): get_employee_count DENIED'),
]

# --- Get tokens for both users ---
tokens = {
    'user-a': get_access_token('user-a', os.environ['USER_A_PASSWORD']),
    'user-b': get_access_token('user-b', os.environ['USER_B_PASSWORD']),
}

# --- Run tests ---
passed = failed = 0

for user, prompt, should_contain, should_not_contain, desc in TEST_CASES:
    print(f'\n{"\u2500"*60}')
    print(f'  {desc}')
    print(f'  Q: {prompt}')
    start = time.time()
    try:
        result = invoke(prompt, tokens[user])
        elapsed = time.time() - start
        print(f'  A: [{result["status"]}] ({elapsed:.1f}s) {result["body"][:200].replace(chr(10), " ")}')
        if check(result, should_contain, should_not_contain):
            print('  \u2705 PASS')
            passed += 1
        else:
            print(f'  \u274c FAIL')
            failed += 1
    except Exception as e:
        print(f'  \u274c FAIL \u2014 {e}')
        failed += 1

print(f'\n{"\u2550"*60}')
print(f'  Results: {passed} passed, {failed} failed, {passed+failed} total')
print(f'{"\u2550"*60}')

## Cleanup

Destroy the CDK stack to avoid ongoing charges.

In [ ]:
!cd .. && PATH=$PWD/.venv/bin:$HOME/.local/bin:/usr/local/bin:/opt/homebrew/bin:/usr/bin:$PATH npx --yes cdk destroy --force